# 핵심 키워드 추출 (Keyword Extraction)

In [ ]:
# # Mecab 설치
# !apt-get update
# !apt-get install g++ openjdk-8-jdk
# !pip3 install konlpy JPype1-py3
# !bash <(curl -s https://raw.githubusercontent.com/konlpy/konlpy/master/scripts/mecab.sh)

In [ ]:
# # mecab-python의 버전 오류로 인해 아래 패키지를 설치하면 코랩에서 Mecab을 사용가능
# !pip install mecab-python3

In [ ]:
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

# # 패키지 업그레이드로 인한 에러 처리를 위해 라이브러리 버전 Downgrade
# !pip install 'networkx<2.7'

# # 런타임 재시작 !

In [2]:
import matplotlib.pyplot as plt
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

# 1.1 TF-IDF 활용 핵심키워드 추출

## 실습 1. sklearn 활용

In [ ]:
%pip install requests beautifulsoup4

In [4]:
import requests
from bs4 import BeautifulSoup

def get_news_by_url(url):
    res = requests.get(url, headers = {
        'User-Agent': 'Mozilla/5.0'
    })
    bs = BeautifulSoup(res.content, 'html.parser')

    title = bs.select_one('#title_area').text #제목
    content = bs.select_one('#contents').get_text().replace('\n', " ") #본문
    content = content.replace("// flash 오류를 우회하기 위한 함수 추가 function _flash_removeCallback() {}", "")
    return  content.strip()

docs = []
docs.append( get_news_by_url('https://news.naver.com/main/read.nhn?mode=LSD&mid=sec&sid1=105&oid=018&aid=0004430108') )
docs.append( get_news_by_url('https://news.naver.com/main/read.nhn?mode=LSD&mid=sec&sid1=101&oid=001&aid=0011614790') )
docs.append( get_news_by_url('https://news.naver.com/main/read.nhn?mode=LSD&mid=sec&sid1=102&oid=014&aid=0004424362') )
docs.append( get_news_by_url('https://news.naver.com/main/read.nhn?mode=LSD&mid=sec&sid1=101&oid=119&aid=0002402191') )
docs.append( get_news_by_url('https://news.naver.com/main/read.nhn?mode=LSD&mid=sec&sid1=101&oid=030&aid=0002882728') )
len(docs)

5

In [ ]:
docs

### 1) 전처리

In [9]:
from konlpy.tag import Mecab
mecab = Mecab('C:/mecab/mecab-ko-dic')
# from konlpy.tag import Komoran
# komoran = Komoran()

preprocessed_docs = []
for doc in docs :
    # 명사와 동사만으로 문서 전처리
    preprocessed_docs.append(' '.join([token[0] for token in mecab.pos(doc) if token[1][0] in ['N', 'V']]))
preprocessed_docs[0][:100]

'과기 정통부 일 유영민 장관 등 참석 기념행사 년 억 원 투입 여종 데이터 구축 민간 클라우드 통한 외부 연계 체계 개방 강화 데일리 이재운 기자 국가 차원 빅 데이터 활용 시대 '

### 2) TF-IDF 계산

In [10]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_df=0.85, max_features=10000)
word_count_vector = count_vectorizer.fit_transform(preprocessed_docs)
list(count_vectorizer.vocabulary_.keys())[:10]

['과기', '정통부', '유영민', '장관', '참석', '기념행사', '투입', '여종', '구축', '민간']

In [11]:
from sklearn.feature_extraction.text import TfidfTransformer

tfidf_transformer = TfidfTransformer(smooth_idf=True, use_idf=True)
tfidf_transformer.fit(word_count_vector)

TfidfTransformer()

### 3) 핵심키워드 추출

In [12]:
def sort_keywords(keywords):
    return sorted(zip(keywords.col, keywords.data), key=lambda x: (x[1], x[0]), reverse=True)

def extract_keywords(feature_names, sorted_keywords, n=5):
    return [(feature_names[idx], score) for idx, score in sorted_keywords[:n]]

In [13]:
doc = preprocessed_docs[0] # 핵심키워드 추출할 문서 조회

feature_names = count_vectorizer.get_feature_names_out() # TF-IDF 단어 목록
tf_idf_vector = tfidf_transformer.transform(count_vectorizer.transform([doc])) # 문서의 tf-idf 추출
sorted_keywords = sort_keywords(tf_idf_vector.tocoo()) # TF-IDF를 기준으로 역순 정렬

# 사용자가 지정한 갯수만큼 키워드 추출
keywords = extract_keywords(feature_names, sorted_keywords, 5)

print("\n===== 원문 =====")
print(docs[0][:100])
print("\n=== 핵심키워드 ===")
for k in keywords:
    print(k)


===== 원문 =====
과기정통부, 22일 유영민 장관 등 참석해 기념행사2021년까지 1516억원 투입, 5100여종 데이터 구축민간 클라우드 통한 외부연계체계도.."개방성 강화"[이데일리 이재운 기자

=== 핵심키워드 ===
('플랫', np.float64(0.2405894811370666))
('데일리', np.float64(0.2405894811370666))
('계획', np.float64(0.2062195552603428))
('정통부', np.float64(0.17184962938361903))
('과기', np.float64(0.17184962938361903))


In [ ]:
tf_idf_vector.tocoo().data


---


## 실습 2. gensim 활용


### 1) 전처리

In [ ]:
from konlpy.tag import Mecab
mecab = Mecab()

preprocessed_docs = []
for doc in docs :
    # 명사와 동사만으로 문서 전처리
    preprocessed_docs.append(' '.join([token[0] for token in mecab.pos(doc) if token[1][0] in ['N', 'V']]))
preprocessed_docs[0][:100]

### 2) TF-IDF 계산

In [ ]:
from gensim.models import TfidfModel
from gensim.corpora import Dictionary

document_ls = [doc.split() for doc in preprocessed_docs]
dct = Dictionary(document_ls) # 인덱스(key) - 단어(value) 인 딕셔너리 생성
corpus = [dct.doc2bow(doc) for doc in document_ls] # 각 문서에 포함된 단어를 인덱스로 변환하여 corpus 생성
tfidf = TfidfModel(corpus) # TF-IDF 산출

### 3) 핵심키워드 추출

In [ ]:
def sort_keywords(tfidf):
    return sorted(tfidf, key=lambda x: (x[1], x[0]), reverse=True)

def extract_keywords(feature_names, sorted_keywords, n=5):
    return [(feature_names[idx], score) for idx, score in sorted_keywords[:n]]

In [ ]:
doc = corpus[0]

sorted_keywords = sort_keywords(tfidf[doc]) # TF-IDF를 기준으로 역순 정렬

# 사용자가 지정한 갯수만큼 키워드 추출
keywords = extract_keywords(dct, sorted_keywords, 5)

print("\n=== 핵심키워드 ===")
for k in keywords:
    print(k)

In [ ]:
tfidf[doc]



---



# 1.2 Textrank
https://web.eecs.umich.edu/~mihalcea/papers/mihalcea.emnlp04.pdf

<img src="https://3.bp.blogspot.com/-yp0Lr3ec5EY/XIs6znCcO_I/AAAAAAAAAPY/xtZxe_OYtH0xeuWsp4Qd4DQrunGMpVQmQCLcBGAs/s640/keyword-extraction-textrank.png" />

In [ ]:
# 그래프 자료구조 표현방식들
scores_nodes = {
    '바나나': 1.0,
    '사과': 1.0,
    '파인애플': 1.0,
    '딸기': 1.0
}
edges = {
    '바나나': ['사과'],
    '사과': ['바나나'],
    '파인애플': [],
    '딸기': []
}

nodes = ['바나나', '사과', '파인애플', '딸기']
word2idx = {}

[
    [1, ], # 바나나
    [0, ], # 사과
]

# 데이터 과학쪽
# np_matrix
array([
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
    [0, 0, 0, 0],
])

# sparse matrix
[
    (r, c, v) 
]

## 실습 1. 행렬 활용


### 1) 토큰화 (Tokenization)

분석 텍스트 정제

### 2) Unique한 토큰 목록 생성

그래프 생성을 위해서 Unique한 토큰 목록 생성

In [14]:
doc = '딸기 바나나 사과 딸기 파인애플'

tokens = doc.split()
unique_tokens = set(tokens)
word2idx = {t: i for i, t in enumerate(unique_tokens)}

print(tokens)
print(word2idx)

['딸기', '바나나', '사과', '딸기', '파인애플']
{'딸기': 0, '파인애플': 1, '바나나': 2, '사과': 3}


### 3) 그래프 생성 (weighted edge 계산)

*   TextRank는 그래프 기반 모델
*   각 단어(토큰)은 그래프의 노드(vertex)
*   weighted_edge 행렬은 노드간 가중치 정보를 담고 있음
*   weighted_edge[i][j] 는 i번째 단어와 j번째 단어의 가중치를 의미
*   weighted_edge[i][j] 가 0인 경우는 노드간 연결이 없음을 의미
*   모든 노드는 1로 초기화

In [ ]:
import numpy as np

# weighted_edges = np.zeros((len(word2idx), len(word2idx)))
weighted_edges = np.eye(len(word2idx)) * 0
# edges = [[0] * len(word2idx) for _ in word2idx]
# edges = [[]  for _ in word2idx]

window_size = 2

for i in range(len(tokens) - window_size + 1):
    temp = tokens[i:i+window_size]
    for ti, token1 in enumerate(temp):
        for token2 in temp[ti+1:]:
            idx_1 = word2idx[token1]
            idx_2 = word2idx[token2]
            weighted_edges[idx_1, idx_2] = 1
            weighted_edges[idx_2, idx_1] = 1

            # edges[idx_1].append(idx_2)
            # edges[idx_2].append(idx_1)

# def make_graph(tokens, window_size):
#     return weighted_edges

scores = np.ones(len(word2idx))
# 정규화
weighted_edges /= weighted_edges.sum(axis=1).reshape(-1, 1)

print(word2idx)
scores, weighted_edges
# print(*edges, sep='\n')

{'딸기': 0, '파인애플': 1, '바나나': 2, '사과': 3}


(array([1., 1., 1., 1.]),
 array([[0.        , 0.33333333, 0.33333333, 0.33333333],
        [1.        , 0.        , 0.        , 0.        ],
        [0.5       , 0.        , 0.        , 0.5       ],
        [0.5       , 0.        , 0.5       , 0.        ]]))

### 4) 각 노드의 score계산
각 노드와 연결된 weighted edge의 값을 합산

In [ ]:
epochs = 30
damping_factor = 0.85
threshold = 0.001

for epoch in range(epochs):
    temp = scores.copy()
    edges = scores.reshape(-1, 1) * weighted_edges
    scores = (1 - damping_factor) + damping_factor * edges.sum(axis=0)
    
    if threshold > np.abs(scores - temp).sum():
        print('조기 종료', epoch)
        break
scores

조기 종료 16


array([1.46711278, 0.56555653, 0.98366534, 0.98366534])

In [123]:
edges = scores.reshape(-1, 1) * weighted_edges
scores = (1 - damping_factor) + damping_factor * edges.sum(axis=0)
scores

array([1.46694347, 0.56563398, 0.98371127, 0.98371127])

In [192]:
edges = scores @ weighted_edges
scores = (1 - damping_factor) + damping_factor * edges
scores

array([1.46694347, 0.56563398, 0.98371127, 0.98371127])

### 5) 핵심 단어 추출

In [213]:
# (-scores).argsort()
idx2word = {i: t for t, i in word2idx.items()}


n = 3
for idx, score in sorted(enumerate(scores), key=lambda x: -x[1])[:n]:
    print(idx2word[idx], score)

# [idx2word[idx] for idx in scores.argsort()[::-1]]

딸기 1.4671127810922537
바나나 0.9836653421526768
사과 0.9836653421526768


In [ ]:
class TextRank:
    def __init__(self):
        pass

    def fit(self):
        ...

    def get_keywords(n):
        ...


---

## 실습 2. 그래프 활용

In [214]:
import requests
from bs4 import BeautifulSoup

def get_news_by_url(url):
    res = requests.get(url, headers = {
        'User-Agent': 'Mozilla/5.0'
    })
    bs = BeautifulSoup(res.content, 'html.parser')

    title = bs.select_one('#title_area').text #제목
    content = bs.select_one('#contents').get_text().replace('\n', " ") #본문
    content = content.replace("// flash 오류를 우회하기 위한 함수 추가 function _flash_removeCallback() {}", "")
    return content.strip()

doc = get_news_by_url('https://news.naver.com/main/read.nhn?mode=LSD&mid=sec&sid1=105&oid=018&aid=0004430108')
doc[:50]

'과기정통부, 22일 유영민 장관 등 참석해 기념행사2021년까지 1516억원 투입, 510'

### 1) 토큰화 (Tokenization)

분석 텍스트 정제

In [216]:
from konlpy.tag import Mecab
mecab = Mecab('C:/mecab/mecab-ko-dic')

nodes = [token for token in mecab.pos(doc) if token[1] in ['NNG','NNP']] #NNG, NNP를 스코어 계산 대상(노드)로 제한
tokens = [token for token in mecab.pos(doc)] #탐색할 토큰 전체

### 2) 그래프 생성 (weighted edge 계산)

*   TextRank는 그래프 기반 모델
*   각 단어(토큰)은 그래프의 노드(vertex) 없음을 의미
*   모든 노드는 1로 초기화

In [ ]:
%pip install networkx

In [219]:
import numpy as np
import math
import networkx as nx

# 윈도내 동시 등장한 토큰으로 그래프를 생성
def connect(nodes, tokens) :
    window_size = 2 # coocurrence를 판단하기 위한 window 사이즈 설정

    edges = []
    for window_start in range(0,(len(tokens)-window_size+1)):
        window = tokens[window_start:window_start+window_size]
        for i in range(window_size) :
            for j in range(i+1, window_size) :
                if (window[i] in nodes) & (window[j] in nodes) : #윈도 내 단어가 노드에 속하는 경우만 엣지로 연결
                    edges.append((window[i], window[j]))
                    print((window[i], window[j]))
    return edges

# tokens = ['딸기', '바나나', '사과', '딸기', '파인애플']
# nodes = ['바나나', '사과', '파인애플', '딸기']

graph=nx.diamond_graph()
graph.clear()
graph.add_nodes_from(list(set(nodes))) #node 등록
graph.add_edges_from(connect(nodes, tokens)) #edge 연결

(('과기', 'NNG'), ('정통부', 'NNG'))
(('유영민', 'NNP'), ('장관', 'NNG'))
(('여종', 'NNG'), ('데이터', 'NNG'))
(('데이터', 'NNG'), ('구축', 'NNG'))
(('구축', 'NNG'), ('민간', 'NNG'))
(('민간', 'NNG'), ('클라우드', 'NNP'))
(('외부', 'NNG'), ('연계', 'NNG'))
(('연계', 'NNG'), ('체계', 'NNG'))
(('데일리', 'NNP'), ('이재운', 'NNP'))
(('이재운', 'NNP'), ('기자', 'NNG'))
(('국가', 'NNG'), ('차원', 'NNG'))
(('빅', 'NNG'), ('데이터', 'NNG'))
(('데이터', 'NNG'), ('활용', 'NNG'))
(('활용', 'NNG'), ('시대', 'NNG'))
(('산업', 'NNG'), ('창출', 'NNG'))
(('기존', 'NNG'), ('산업', 'NNG'))
(('데이터', 'NNG'), ('경제', 'NNG'))
(('과학', 'NNG'), ('기술', 'NNG'))
(('기술', 'NNG'), ('정보', 'NNG'))
(('정보', 'NNG'), ('통신부', 'NNG'))
(('서울', 'NNP'), ('중구', 'NNP'))
(('중구', 'NNP'), ('대한', 'NNP'))
(('대한', 'NNP'), ('상공', 'NNG'))
(('상공', 'NNG'), ('회의소', 'NNG'))
(('데이터', 'NNG'), ('생태', 'NNG'))
(('기반', 'NNG'), ('마련', 'NNG'))
(('빅', 'NNG'), ('데이터', 'NNG'))
(('데이터', 'NNG'), ('플랫', 'NNG'))
(('플랫', 'NNG'), ('폼', 'NNG'))
(('출범식', 'NNG'), ('행사', 'NNG'))
(('유영민', 'NNP'), ('과기', 'NNG'))
(('과기', 'NNG'), ('정통부',

### 3) 스코어 계산 및 핵심키워드 추출

In [220]:
scores = nx.pagerank(graph) #pagerank 계산
rank = sorted(scores.items(), key=lambda x: x[1], reverse=True) #score 역순 정렬
print("\n=== 핵심키워드 ===")
rank[:5]


=== 핵심키워드 ===


[(('데이터', 'NNG'), 0.03296894818378009),
 (('정보', 'NNG'), 0.012655813562979047),
 (('활용', 'NNG'), 0.011214052395668282),
 (('플랫', 'NNG'), 0.009222261313758837),
 (('주요', 'NNG'), 0.00913375727303187)]

In [222]:
%pip install pydot


  Using cached pydot-4.0.1-py3-none-any.whl.metadata (11 kB)
Using cached pydot-4.0.1-py3-none-any.whl (37 kB)


In [ ]:
import networkx as nx
from IPython.core.display import Image
from networkx.drawing.nx_pydot import to_pydot

d = to_pydot(graph)
d.set_dpi(600)
d.set_rankdir("LR")
Image(d.create_png(), width=600)

---

## 실습 3. TextRank 핵심 구 추출

### 1) 불용어를 기준으로 구 추출

In [224]:
phrases = []
phrase = ' '

for word in tokens:
    if word[1][0] != 'N':
        if phrase != ' ':
            phrases.append(phrase.strip())
        phrase= ' '
    else:
        phrase+=word[0]
        phrase+= ' '

print(phrases)

['과기 정통부', '일 유영민 장관 등 참석', '기념행사', '년', '억 원 투입', '여종 데이터 구축 민간 클라우드', '외부 연계 체계', '개방', '강화', '데일리 이재운 기자', '국가 차원', '빅 데이터 활용 시대', '산업 창출', '기존 산업', '변화', '혁신', '장', '센터', '문', '개 분야', '데이터 경제', '발전', '정부', '청사진', '현실', '구현', '데', '계획', '일 과학 기술 정보 통신부', '서울 중구 대한 상공 회의소', '데이터 생태', '조성', '혁신', '장', '기반 마련', '빅 데이터 플랫 폼', '센터', '출범식 행사', '개최', '유영민 과기 정통부 장관', '노웅래 국회 과학 기술 정보 방송 통신 위원회 위원장 등', '명', '참가', '개 분야', '개 센터', '년 간', '억 원 투입 이미지', '픽사 베이 빅 데이터', '데이터 활용', '혁신', '장', '문재', '정부', '경제 성장 핵심 요소 중 하나', '문재인 대통령', '데이터 활용', '이', '정보 보호', '보안', '중요', '강조', '맥락 속', '빅 데이터 센터', '공공', '민간', '협업', '활용', '양질', '데이터', '생산', '구축', '플랫 폼', '이', '수집', '분석', '유통', '역할', '담당', '과기 정통부', '분야', '플랫 폼', '개소', '이', '연계', '기관', '센터', '개소', '구축', '데', '년', '억 원', '투입', '계획', '올해', '억 원 규모', '사업', '추진', '대상 분야', '금융', '카드', '환경', '한국 수자원 공사', '문화', '한국 문화 정보원', '교통', '한국 교통 연구원', '헬 스케어', '국립암센터', '유통', '소비', '매일 방송', '통신', '중소기업', '비즈', '지역 경제', '경기도 청', '산림', '한국 임업', '흥원', '등', '차 공모', '개 빅 데이

In [225]:
unique_phrases = []

for phrase in phrases:
    if phrase not in unique_phrases:
        unique_phrases.append(phrase)

print(unique_phrases)

['과기 정통부', '일 유영민 장관 등 참석', '기념행사', '년', '억 원 투입', '여종 데이터 구축 민간 클라우드', '외부 연계 체계', '개방', '강화', '데일리 이재운 기자', '국가 차원', '빅 데이터 활용 시대', '산업 창출', '기존 산업', '변화', '혁신', '장', '센터', '문', '개 분야', '데이터 경제', '발전', '정부', '청사진', '현실', '구현', '데', '계획', '일 과학 기술 정보 통신부', '서울 중구 대한 상공 회의소', '데이터 생태', '조성', '기반 마련', '빅 데이터 플랫 폼', '출범식 행사', '개최', '유영민 과기 정통부 장관', '노웅래 국회 과학 기술 정보 방송 통신 위원회 위원장 등', '명', '참가', '개 센터', '년 간', '억 원 투입 이미지', '픽사 베이 빅 데이터', '데이터 활용', '문재', '경제 성장 핵심 요소 중 하나', '문재인 대통령', '이', '정보 보호', '보안', '중요', '강조', '맥락 속', '빅 데이터 센터', '공공', '민간', '협업', '활용', '양질', '데이터', '생산', '구축', '플랫 폼', '수집', '분석', '유통', '역할', '담당', '분야', '개소', '연계', '기관', '억 원', '투입', '올해', '억 원 규모', '사업', '추진', '대상 분야', '금융', '카드', '환경', '한국 수자원 공사', '문화', '한국 문화 정보원', '교통', '한국 교통 연구원', '헬 스케어', '국립암센터', '소비', '매일 방송', '통신', '중소기업', '비즈', '지역 경제', '경기도 청', '산림', '한국 임업', '흥원', '등', '차 공모', '개 빅 데이터 센터', '선정', '다음 달', '일', '개', '추가 선정', '지원', '운영', '데이터 생태계', '기업', '경쟁력', '제고', '수행', '주요 활용 전략', '사례', '빅 데이터 활용', '신

### 2) 각 구의 Score 계산

앞서 산출한 각 단어별 점수를 합산

In [226]:
vocabs = dict((r[0][0], r[1]) for r in rank)
phrase_scores = []
keywords = []

for phrase in unique_phrases:
    phrase_score = 0
    for word in phrase.split():
        if word in vocabs.keys():
            phrase_score+=vocabs[word]

    phrase_scores.append((phrase, phrase_score))

for phrase_score in phrase_scores[:10]:
    print(f"keyword: {phrase_score[0]}, Score : {phrase_score[1]}")

keyword: 과기 정통부, Score : 0.009878440032129062
keyword: 일 유영민 장관 등 참석, Score : 0.008690478798287035
keyword: 기념행사, Score : 0.0006327778949588702
keyword: 년, Score : 0
keyword: 억 원 투입, Score : 0.004218519299725797
keyword: 여종 데이터 구축 민간 클라우드, Score : 0.05019729098459901
keyword: 외부 연계 체계, Score : 0.01265555789917739
keyword: 개방, Score : 0.0006327778949588702
keyword: 강화, Score : 0.0006327778949588702
keyword: 데일리 이재운 기자, Score : 0.014411017533996647


### 3) 각 구를 Score로 정렬하여 핵심 구 추출

In [227]:
sorted_phrase_score = sorted(phrase_scores, key=lambda tup: tup[1], reverse=True)
sorted_phrase_score[:10]

[('여종 데이터 구축 민간 클라우드', 0.05019729098459901),
 ('민간 사이 데이터 파일 형식 등', 0.048946326911462584),
 ('빅 데이터 활용 시대', 0.04851225217227742),
 ('빅 데이터 플랫 폼', 0.046499633123186515),
 ('빅 데이터 활용', 0.046291121290244264),
 ('노웅래 국회 과학 기술 정보 방송 통신 위원회 위원장 등', 0.046078931169150975),
 ('데이터 활용', 0.04418300057944838),
 ('여종 신규 데이터', 0.04399287251297052),
 ('여종 데이터 구축', 0.042230708688984736),
 ('픽사 베이 빅 데이터', 0.042137337577637095)]



---



## 실습 4. gensim 활용

- summarization.keywords(document, words) : document를 요약하여 words만큼 키워드를 추출

In [ ]:
# %pip uninstall gensim -y
%pip install gensim==3.8.3

In [236]:
import gensim
gensim.__version__

'4.4.0'

In [228]:
from gensim.summarization import keywords
keywords(doc, words=5).split('\n')

ModuleNotFoundError: No module named 'gensim.summarization'